# Trading Strategy Analysis
**Date:** [Insert Today's Date]  
**Author:** [Your Name]  
**Description:** This notebook analyzes price inefficiencies in the market using a quantitative trading strategy. The goal is to develop models for trading and manage risk effectively.


In [9]:
# Standard libraries
import os
import time
import datetime

# Data science
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# ML and preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, precision_score, mean_squared_error, r2_score;
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor;
from sklearn.model_selection import train_test_split;
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression


# Concurrency and I/O
import concurrent.futures
import joblib
import yfinance as yf
from functools import lru_cache;



In [10]:
# Step 1: Extract the Data
# Fetch historical data for a stock ticker with caching

CACHE_DIR = "data_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def fetch_ticker_data(ticker, period='5y', interval='1d', start=None, end=None):
    date_suffix = f"{start}_{end}" if start and end else period
    cache_file = os.path.join(CACHE_DIR, f"{ticker}_{date_suffix}.pkl")

    if os.path.exists(cache_file):
        print(f"Loading cached data for {ticker}")
        return joblib.load(cache_file)

    try:
        df = yf.download(ticker, period=period, interval=interval, start=start, end=end)
        df['Return'] = df['Adj Close'].pct_change()
        df['Ticker'] = ticker
        joblib.dump(df, cache_file)
        return df
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return None

def fetch_multiple_tickers(tickers=['AAPL', 'TSLA'], period='5y', interval='1d', start=None, end=None):
    start_time = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(lambda t: fetch_ticker_data(t, period, interval, start, end), tickers))

    df = pd.concat([r for r in results if r is not None], axis=0)
    fetch_time = time.time() - start_time
    return df, fetch_time



In [ ]:
# Step 2: Preprocess the Data 


# ----------------------------
# Custom Transformers
# ----------------------------

class StockFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, lags=[1, 5], rolling_windows=[5, 10]):
        self.lags = lags
        self.rolling_windows = rolling_windows

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        
        # Sort data to ensure correct rolling/lagging
        df = df.sort_values(by=['Ticker', 'Date'])

        # Lag features (previous returns)
        for lag in self.lags:
            df[f'Return_Lag_{lag}'] = df.groupby('Ticker')['Return'].shift(lag)

        # Rolling mean and std (volatility) on returns
        for window in self.rolling_windows:
            df[f'RollingMean_{window}'] = df.groupby('Ticker')['Return'].transform(lambda x: x.rolling(window).mean())
            df[f'RollingStd_{window}'] = df.groupby('Ticker')['Return'].transform(lambda x: x.rolling(window).std())

        # Momentum: Rate of Change
        df['ROC_5'] = df.groupby('Ticker')['Adj Close'].transform(lambda x: x.pct_change(periods=5))
        df['ROC_10'] = df.groupby('Ticker')['Adj Close'].transform(lambda x: x.pct_change(periods=10))

        # Time-based features
        df['DayOfWeek'] = df['Date'].dt.dayofweek
        df['Month'] = df['Date'].dt.month

        # Drop rows with NaNs from lagging/rolling
        df = df.dropna()

        return df


class StockScaler(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_scale=None):
        self.columns_to_scale = columns_to_scale
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        if self.columns_to_scale is None:
            self.columns_to_scale = X.select_dtypes(include=[np.number]).columns.tolist()
        self.scaler.fit(X[self.columns_to_scale])
        return self

    def transform(self, X):
        X_scaled = X.copy()
        X_scaled[self.columns_to_scale] = self.scaler.transform(X_scaled[self.columns_to_scale])
        return X_scaled


# ----------------------------
# Preprocessing Pipeline
# ----------------------------

def preprocess_stock_data(df_raw):
    """
    Main function to preprocess stock data.

    Assumes df has columns: ['Date', 'Adj Close', 'Return', 'Ticker']
    """
    df = df_raw.copy()
    
    # Ensure 'Date' is datetime
    df['Date'] = pd.to_datetime(df['Date'])

    # Drop rows with missing values in core fields (e.g., due to bad API fetch)
    df = df.dropna(subset=['Adj Close', 'Return'])

    # Sort data
    df = df.sort_values(by=['Ticker', 'Date'])

    # Select relevant columns
    df = df[['Date', 'Ticker', 'Adj Close', 'Return']]

    # Build pipeline
    pipeline = Pipeline([
        ('feature_engineering', StockFeatureEngineer()),
        ('scaling', StockScaler(columns_to_scale=None)),  # Auto-select numeric columns
    ])

    df_processed = pipeline.fit_transform(df)

    return df_processed


In [12]:
# Step 3: Feature/Target Prep + Split

def prepare_features_and_target(df_processed, split_date='2022-01-01'):
    """
    Prepare X, y and split into train/test sets (time-aware).
    Target: next-day return direction (classification)
    """
    df = df_processed.copy()

    # Define target (next day up = 1, down = 0)
    df['Target'] = np.where(df['Return'].shift(-1) > 0, 1, 0)

    # Drop last row for each ticker (no future return available)
    df = df.groupby('Ticker').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)

    # Define features (exclude identifiers & raw return)
    feature_cols = [col for col in df.columns if col not in ['Date', 'Ticker', 'Return', 'Target']]
    X = df[feature_cols]
    y = df['Target']

    # Time-aware split
    train_mask = df['Date'] < split_date
    test_mask = df['Date'] >= split_date

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    return X_train, X_test, y_train, y_test, df

In [13]:
# Step 4: Model Training and Baseline Evaluation

def train_and_evaluate(X_train, X_test, y_train, y_test):
    
    results = {}
    
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    }
    
    for name, model in models.items():
        start_time = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start_time
        y_pred = model.predict(X_test)
        
    results[name] = {
            'model': model,     
            'train_time': train_time,
            'mse': mean_squared_error(y_test, y_pred),
            'r2': r2_score(y_test, y_pred),
            'accuracy': accuracy_score(y_test, np.round(y_pred)),
            'precision': precision_score(y_test, np.round(y_pred), zero_division=0),
            'recall': recall_score(y_test, np.round(y_pred), zero_division=0),
            'f1': f1_score(y_test, np.round(y_pred), zero_division=0),
            'confusion_matrix': confusion_matrix(y_test, np.round(y_pred)),
            
        }  
            
    return results


In [14]:
# Step 5: Model Optimization and Advanced Evaluation

def optimize_models(X_train, y_train):
    """
    Optimize Logistic Regression and Random Forest using TimeSeriesSplit.
    Returns best estimators and their CV scores.
    """
    results = {}

    # Define time-aware CV
    tscv = TimeSeriesSplit(n_splits=5)

    # -----------------
    # Logistic Regression
    # -----------------
    log_reg = LogisticRegression(max_iter=1000)
    log_params = {
        "C": [0.01, 0.1, 1, 10],
        "penalty": ["l2"],
        "solver": ["lbfgs"]
    }
    log_search = GridSearchCV(
        estimator=log_reg,
        param_grid=log_params,
        cv=tscv,
        scoring="f1",
        n_jobs=-1
    )
    log_search.fit(X_train, y_train)
    results["Logistic Regression"] = {
        "best_model": log_search.best_estimator_,
        "best_params": log_search.best_params_,
        "best_score": log_search.best_score_
    }

    # -----------------
    # Random Forest
    # -----------------
    rf = RandomForestClassifier(random_state=42)
    rf_params = {
        "n_estimators": [100, 200],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }
    rf_search = GridSearchCV(
        estimator=rf,
        param_grid=rf_params,
        cv=tscv,
        scoring="f1",
        n_jobs=-1
    )
    rf_search.fit(X_train, y_train)
    results["Random Forest"] = {
        "best_model": rf_search.best_estimator_,
        "best_params": rf_search.best_params_,
        "best_score": rf_search.best_score_
    }

    return results


def evaluate_best_models(models, X_test, y_test):
    """
    Evaluate tuned models on the test set.
    """
    eval_results = {}

    for name, info in models.items():
        model = info["best_model"]
        y_pred = model.predict(X_test)

        eval_results[name] = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1": f1_score(y_test, y_pred)
        }

    return eval_results


In [15]:
# ========================================
# Step 6: Backesting and Strategy Simulation
# ========================================

def backtest_strategy(model, X_test, y_test, df_test, risk_free_rate=0.0):
    """
    Backtest trading strategy using model predictions.
    
    Args:
        model: trained classifier with predict()
        X_test: test features
        y_test: true labels (not directly used in returns but for comparison)
        df_test: original dataframe aligned with X_test (must include 'Return' column)
        risk_free_rate: daily risk-free rate for Sharpe ratio (default 0)
    
    Returns:
        results: dict with performance metrics
        df_bt: dataframe with signals, returns, and equity curve
    """
    df_bt = df_test.copy().reset_index(drop=True)

    # Predictions (signals)
    df_bt["Signal"] = model.predict(X_test)  # 1 = long, 0 = short

    # Map signals to positions: 1 = long, -1 = short
    df_bt["Position"] = df_bt["Signal"].replace({0: -1, 1: 1})

    # Strategy daily returns = position * actual return
    df_bt["Strategy_Return"] = df_bt["Position"] * df_bt["Return"]

    # Cumulative returns
    df_bt["Cumulative_Strategy"] = (1 + df_bt["Strategy_Return"]).cumprod()
    df_bt["Cumulative_BuyHold"] = (1 + df_bt["Return"]).cumprod()

    # Performance metrics
    total_return = df_bt["Cumulative_Strategy"].iloc[-1] - 1
    avg_daily_return = df_bt["Strategy_Return"].mean()
    std_daily_return = df_bt["Strategy_Return"].std()

    sharpe_ratio = np.nan
    if std_daily_return != 0:
        sharpe_ratio = (avg_daily_return - risk_free_rate) / std_daily_return * np.sqrt(252)

    # Max drawdown
    cum_max = df_bt["Cumulative_Strategy"].cummax()
    drawdown = (df_bt["Cumulative_Strategy"] - cum_max) / cum_max
    max_drawdown = drawdown.min()

    results = {
        "Total Return (%)": round(total_return * 100, 2),
        "Sharpe Ratio": round(sharpe_ratio, 3),
        "Max Drawdown (%)": round(max_drawdown * 100, 2),
        "Final Equity": round(df_bt["Cumulative_Strategy"].iloc[-1], 3)
    }

    return results, df_bt

In [16]:
# ========================================
# Step 7: Final Evaluation & Reporting
# ========================================

import pandas as pd
import matplotlib.pyplot as plt

def compare_models(test_results, backtest_results):
    """
    Combine classification metrics and backtest results into a summary DataFrame.
    
    Args:
        test_results: dict of classification metrics (from Step 5 evaluate_best_models)
        backtest_results: dict of backtest performance (from Step 6 backtest_strategy)
    
    Returns:
        summary_df: DataFrame with combined results
    """
    rows = []

    for model_name in test_results.keys():
        row = {
            "Model": model_name,
            "Accuracy": round(test_results[model_name]["Accuracy"], 3),
            "Precision": round(test_results[model_name]["Precision"], 3),
            "Recall": round(test_results[model_name]["Recall"], 3),
            "F1": round(test_results[model_name]["F1"], 3),
            "Total Return (%)": backtest_results[model_name]["Total Return (%)"],
            "Sharpe Ratio": backtest_results[model_name]["Sharpe Ratio"],
            "Max Drawdown (%)": backtest_results[model_name]["Max Drawdown (%)"],
            "Final Equity": backtest_results[model_name]["Final Equity"],
        }
        rows.append(row)

    summary_df = pd.DataFrame(rows)
    return summary_df


def plot_equity_curves(backtest_dfs):
    """
    Plot equity curves for multiple models vs Buy & Hold.
    
    Args:
        backtest_dfs: dict of DataFrames returned from backtest_strategy
    """
    plt.figure(figsize=(10, 6))
    for model_name, df_bt in backtest_dfs.items():
        plt.plot(df_bt["Date"], df_bt["Cumulative_Strategy"], label=f"{model_name} Strategy")
    # All should have the same Buy & Hold
    sample_df = list(backtest_dfs.values())[0]
    plt.plot(sample_df["Date"], sample_df["Cumulative_BuyHold"], label="Buy & Hold", linestyle="--", color="black")
    plt.title("Equity Curves: Models vs Buy & Hold")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Returns")
    plt.legend()
    plt.show()


# ----------------------------
# Example Usage (after Step 6)
# ----------------------------
if __name__ == "__main__":
    # Evaluate each best model with backtesting
    backtest_results = {}
    backtest_dfs = {}

    for model_name, info in best_models.items():
        model = info["best_model"]
        df_test = df_final[df_final['Date'] >= '2022-01-01'].reset_index(drop=True)

        results, df_bt = backtest_strategy(model, X_test, y_test, df_test)
        backtest_results[model_name] = results
        backtest_dfs[model_name] = df_bt

    # Step 7a: Combine results into summary table
    summary_df = compare_models(test_results, backtest_results)
    print("\n===== Final Model Comparison =====")
    print(summary_df)

    # Step 7b: Plot equity curves
    plot_equity_curves(backtest_dfs)


NameError: name 'best_models' is not defined

## Conclusion
This notebook demonstrates a simple quantitative trading strategy based on moving averages. Future work could involve optimizing the parameters or incorporating additional data sources for improved predictions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Step 1: Fetch the Data
tickers = ['AAPL', 'GOOGL', 'AMZN', 'TSLA']
data = yf.download(tickers, start='2020-01-01', end='2023-01-01')

# Step 2: Calculate Adjusted Close Returns
returns = data['Adj Close'].pct_change()

# Step 3: Calculate Moving Averages
moving_averages_20 = data['Adj Close'].rolling(window=20).mean()
moving_averages_50 = data['Adj Close'].rolling(window=50).mean()

# Develop a Trading Strategy
# Generate signals
data['Signal'] = 0

# Ensure we compare the moving averages correctly
data['Signal'][20:] = (moving_averages_20[20:] > moving_averages_50[20:]).astype(int)

# Create the Position column
data['Position'] = data['Signal'].diff()

# Check if 'Position' column was created
print("DataFrame columns:", data.columns)

# Evaluate the Strategy
if 'Position' in data.columns:
    # Calculate total returns
    total_return = (data['Adj Close'].iloc[-1] - data['Adj Close'].iloc[0]) / data['Adj Close'].iloc[0]

    # Calculate number of trades
    num_trades = data['Position'].dropna().count()

    # Calculate returns for each trade
    data['Trade Return'] = data['Adj Close'].pct_change().shift(-1) * data['Signal'].shift(1)

    # Calculate average return per trade
    avg_return_per_trade = data['Trade Return'].mean()

    # Print evaluation results
    print(f'Total Return: {total_return:.2%}')
    print(f'Number of Trades: {num_trades}')
    print(f'Average Return per Trade: {avg_return_per_trade:.2%}')
else:
    print("The 'Position' column does not exist in the DataFrame.")


[*********************100%***********************]  4 of 4 completed
C:\Users\bobby\AppData\Local\Temp\ipykernel_2532\3921255009.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Signal'][20:] = (moving_averages_20[20:] > moving_averages_50[20:]).astype(int)


ValueError: setting an array element with a sequence.